# Copyright Rayyan Hodges, TAFE NSW, Gelos Enterprises, Indigo Community Services and Health Hub © 2026

# Contact: rayyan.hodges@studytafensw.edu.au, rayyan.hodges@gelosmail.com.au indigoCSHH@hmail.com

# Program Name: HopsitalCancerClassification.ipynb

# Purpose: To train and evaluate an ML model to automate the task of classifying clients as high, medium or low risk of developing cancer.

In [1]:
# Import required libraries
import joblib
import numpy
import pandas as pd
import sklearn # importing the scikit-learn project, different name due to syntax rules 
import scipy
import threadpoolctl

In [3]:
# Set appropriate display options in Pandas
pd.set_option('display.max_rows', 10000)
pd.set_option('display.max_columns', 10000)
pd.set_option('display.width', 10000)

# Identify any datapoints with missing labels and remove them

In [4]:
# Load the data set into memory and define it for further analysis.
df = pd.read_csv("data.csv")
# Check for missing values in the target column, count them, and display them in a total number.
df['cancer_risk'].isnull().sum()

np.int64(10)

In [5]:
# Remove the affected rows with missing data
df = df.dropna(subset=['cancer_risk'])
# Verify removal of said invalid data.
df['cancer_risk'].isnull().sum()

np.int64(0)

# Categorise ages as required.

In [6]:
# Define age bins and labels for the groups
# Bins meaning boundaries 
bins = [0, 14, 24, 34, 44, 54, 64, 74, 84, 120]
labels = ['0-14', '15-24', '25-34', '35-44', 
          '45-54', '55-64', '65-74', '75-84', '85+']

# Categorise ages into age groups
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

# Display the results in a table
df[['age', 'age_group']].head()


,age,age_group
0,51,45-54
1,92,85+
2,14,0-14
3,71,65-74
4,60,55-64


# Standardise numerical features using appropriate scaling techniques.

In [7]:
# Import the scaling component from the SciKit Learning Library
from sklearn.preprocessing import StandardScaler

# Select numerical features from the datasheet.
numerical_features = ['bmi', 'blood_pressure']

# Initialise scaler component
scaler = StandardScaler()

# Apply scaling and transformation to standardise the numbers.
df[numerical_features] = scaler.fit_transform(df[numerical_features])

# Display results from the scaling and transformation.
df[numerical_features].describe()


,bmi,blood_pressure
count,9.900000e+02,9.900000e+02
mean,1.076580e-17,1.130409e-16
std,1.000505e+00,1.000505e+00
min,-1.679483e+00,-1.699762e+00
25%,-8.839714e-01,-8.598235e-01
50%,6.927133e-04,-1.794981e-02
75%,8.853568e-01,8.703721e-01
max,1.735732e+00,1.772241e+00


# Encode categorical data using appropriate encoding techniques.

In [8]:
# Identify categorical features from the dataset file to be referenced.
categorical_features = ['age_group', 'smoking_history', 'diet', 'exercise_frequency']

# Apply one-hot encoding to convert the text categories into dummy numerical categories (specifically true or false)
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# Print the results to ensure the categorical features have been converted to variables such as true or false correctly.
df_encoded.head()


,age,family_history,alcohol_consumption,bmi,blood_pressure,cholesterol_level,travelled_overseas,number_of_children,cancer_risk,age_group_15-24,age_group_25-34,age_group_35-44,age_group_45-54,age_group_55-64,age_group_65-74,age_group_75-84,age_group_85+,smoking_history_former,smoking_history_never,diet_good,diet_poor,exercise_frequency_low,exercise_frequency_moderate,exercise_frequency_none
0,51,no,low,0.487601,1.629026,226.4,no,5,moderate,False,False,False,True,False,False,False,False,False,True,False,True,False,False,True
1,92,no,moderate,-1.487463,1.396785,170.9,yes,2,low,False,False,False,False,False,False,False,True,False,True,False,True,False,False,False
2,14,no,none,0.528748,0.568459,279.9,yes,4,high,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
3,71,yes,low,1.667153,-0.101170,280.7,yes,0,moderate,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False
4,60,no,none,-1.446316,0.649743,212.8,yes,5,moderate,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True


# Divide and randomise the dataset into training

In [9]:

# Remove rows with missing target labels (specifically, the 'cancer_risk' label, as it has missing data) 
df_encoded = df_encoded.dropna(subset=['cancer_risk'])

#Import the splitting model from the scikit-learn model.
from sklearn.model_selection import train_test_split

# Separate features and target
X = df_encoded.drop('cancer_risk', axis=1)
y = df_encoded['cancer_risk']

# First split: 70% training, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Print the results of said sizes to ensure it is working correctly.
X_train.shape, X_val.shape, X_test.shape


((693, 23), (148, 23), (149, 23))

# Select the most important features in the training dataset that may influence cancer risk, dropping 2 unnecessary features.

In [10]:
# Drop two unnecessary features/columns that aren't relevant to cancer risk evaluation and wouldn't affect if someone would be likely to have cancer.
X_train = X_train.drop(['travelled_overseas', 'number_of_children'], axis=1)
X_val   = X_val.drop(['travelled_overseas', 'number_of_children'], axis=1)
X_test  = X_test.drop(['travelled_overseas', 'number_of_children'], axis=1)

# Print the results to allow the user to determine the changes have worked
X_train.shape


(693, 21)

# Part 3: Train and evaluate ML model - Task 1: Train ML model 1

In [11]:
# Select an appropriate initial model size by setting the number of layers and neurons.


from sklearn.neural_network import MLPClassifier

mlp_model_1 = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)


In [13]:
# BUG FIXING: List any non numerical rows
X_train.dtypes

age                              int64
family_history                     str
alcohol_consumption                str
bmi                            float64
blood_pressure                 float64
cholesterol_level              float64
age_group_15-24                   bool
age_group_25-34                   bool
age_group_35-44                   bool
age_group_45-54                   bool
age_group_55-64                   bool
age_group_65-74                   bool
age_group_75-84                   bool
age_group_85+                     bool
smoking_history_former            bool
smoking_history_never             bool
diet_good                         bool
diet_poor                         bool
exercise_frequency_low            bool
exercise_frequency_moderate       bool
exercise_frequency_none           bool
dtype: object

In [14]:
# BUG FIXING: Check if the text value exists
X_train.select_dtypes(include=['object']).head()

/tmp/ipykernel_82/146862157.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_train.select_dtypes(include=['object']).head()


,family_history,alcohol_consumption
494,yes,none
379,yes,high
220,yes,none
732,yes,high
869,no,low


In [18]:
# BUG FIXING - Rayyan Hodges

# Force all non-numeric columns to be one-hot encoded
X_train = pd.get_dummies(X_train)
X_val   = pd.get_dummies(X_val)
X_test  = pd.get_dummies(X_test)

# Align columns so they match exactly
X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)


In [19]:
# Train model 1 on generated data
mlp_model_1.fit(X_train, y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(64, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",300
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42


# Run the validation data through the ML model.

In [22]:
# Run validation data through the trained model
y_val_pred_1 = mlp_model_1.predict(X_val)
y_val_proba_1 = mlp_model_1.predict_proba(X_val)


# This script outputs nothing by default, so adding a line to display the first ten lines as evidence.
y_val_pred_1[:10]

array(['low', 'high', 'high', 'moderate', 'low', 'high', 'high', 'high',
       'low', 'low'], dtype='<U8')

# Metrics Calculation (Accuracy, ROC-AUC & Confusion matrix) - Rayyan Hodges

In [26]:
# Calculating accuracy

from sklearn.metrics import accuracy_score

accuracy_1 = accuracy_score(y_val, y_val_pred_1)
accuracy_1


0.6216216216216216

In [27]:
# Calculating ROC‑AUC (multi‑class)

from sklearn.metrics import roc_auc_score

roc_auc_1 = roc_auc_score(
    y_val,
    y_val_proba_1,
    multi_class='ovr'
)
roc_auc_1


0.8328379028379028

In [25]:
#Calculating confusion matrix

from sklearn.metrics import confusion_matrix

conf_matrix_1 = confusion_matrix(y_val, y_val_pred_1)
conf_matrix_1

array([[35,  1, 13],
       [ 1, 40,  8],
       [14, 19, 17]])